# Approach 3: draw the box on the image, then ask klein to use it (text + box only)

Instead of erasing the box region (approach 1), we **leave the scene intact and draw a colored rectangle outline on it**, then prompt:
*"Put <object> inside the red box, then remove the box."* No LoRA, no reference image: just FLUX.2 [klein] 4B's own multi-reference/edit ability.

**How it differs from approach 1 (inpainting with a filled box)**
| | Approach 1 (filled box) | Approach 3 (drawn outline) |
|---|---|---|
| What the model sees in the box | solid magenta (nothing to preserve) | the real scene content + a thin outline |
| Pro | box is a strong "fill this" signal | model sees the context under the box, so blending is more natural |
| Con | placeholder color can leak | **outline can leak into the output**, and the box is a weaker constraint (object may ignore/overflow it) |

Because the constraint is soft, this notebook keeps the safety net from v2: **best-of-N with scoring** (leftover outline, overflow, nothing-inserted), a **change-aware blend region** (overflow kept, not clipped), and Poisson/alpha blending. Pixels outside the final region stay identical to the input.

`compare_marks(...)` runs different box styles (red/green/magenta outline) side by side so you can see which one klein follows best on your image.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors pillow matplotlib opencv-python-headless scipy

In [ ]:
import torch
from diffusers import Flux2KleinPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

# Distilled 4B: 4 steps, guidance 1.0 (Apache 2.0, ~13 GB VRAM)
pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=torch.bfloat16)

pipe.to(device)                        # fastest, needs ~13+ GB VRAM
# pipe.enable_model_cpu_offload()      # use this instead if you run out of memory

## Helpers

In [ ]:
import io
import numpy as np
import cv2
from scipy import ndimage as ndi
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch

# ---------- how the box is shown to the model ----------
MARKS = {                       # name -> (outline color, color word used in the prompt)
    "red":     ((255, 0, 0),   "red"),
    "green":   ((0, 255, 0),   "green"),
    "magenta": ((255, 0, 255), "magenta"),
}


def draw_marker(work, wbox, mark, width):
    """Outline drawn INSIDE the bbox edge (so leftover marker pixels are never counted as overflow)."""
    img = work.copy()
    ImageDraw.Draw(img).rectangle([round(v) for v in wbox], outline=MARKS[mark][0], width=width)
    return img


def outline_band(shape_hw, wbox, width):
    """Boolean mask of the drawn outline pixels."""
    outer = box_mask_np(shape_hw, wbox)
    x0, y0, x1, y1 = wbox
    inner = box_mask_np(shape_hw, (x0 + width, y0 + width, x1 - width, y1 - width))
    return outer & ~inner


def scrub_marker(out, work, wbox, mark, width):
    """Safety net: any marker-colored pixel still on the outline band is replaced by the original scene pixel."""
    o = np.array(out)
    w = np.asarray(work)
    band = outline_band(o.shape[:2], wbox, width)
    col = np.array(MARKS[mark][0])
    bad = (np.abs(o.astype(np.int16) - col).sum(2) < 120) & band
    bad = ndi.binary_dilation(bad, iterations=1) & band
    o[bad] = w[bad]
    return Image.fromarray(o), float(bad.sum() / max(band.sum(), 1))


# ---------- prompting ----------
def default_prompt(obj, box_w, box_h, mark):
    ar = box_w / box_h
    shape = "wide, landscape-shaped" if ar > 1.3 else ("tall, portrait-shaped" if ar < 0.77 else "roughly square")
    color = MARKS[mark][1]
    return (f"A {color} rectangular outline is drawn on the image. Add {obj} inside the {color} rectangle: the whole "
            f"{obj} must fit completely inside it, with a small margin, and must not extend beyond it. The rectangle "
            f"is {shape}. Then completely remove the {color} rectangle outline. Match the lighting, colors, motion "
            f"blur, resolution and compression quality of the surrounding image; no drop shadow, no studio look. "
            f"Keep everything else unchanged.")


# ---------- geometry ----------
def get_crop_box(bbox, img_size, context_scale=2.5, min_side=256):
    x0, y0, x1, y1 = bbox
    W, H = img_size
    w, h = x1 - x0, y1 - y0
    cw = min(W, max(w * context_scale, min_side))
    ch = min(H, max(h * context_scale, min_side))
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    left = min(max(cx - cw / 2, 0), W - cw)
    top = min(max(cy - ch / 2, 0), H - ch)
    return tuple(int(round(v)) for v in (left, top, left + cw, top + ch))


def work_size(w, h, target):
    s = target / max(w, h)
    return max(16, round(w * s / 16) * 16), max(16, round(h * s / 16) * 16)


def auto_target(cw, ch, lo=512, hi=1024):
    return int(min(hi, max(lo, max(cw, ch))))


def box_mask_np(shape_hw, box):
    h, w = shape_hw
    m = np.zeros((h, w), bool)
    x0, y0, x1, y1 = [int(round(v)) for v in box]
    m[max(y0, 0):min(y1, h), max(x0, 0):min(x1, w)] = True
    return m


# ---------- candidate analysis ----------
def analyze_candidate(out, work, wbox, mark, width, thr=28):
    """Score a candidate: leftover outline, overflow past the box, and whether anything was added."""
    o = np.asarray(out, np.int16)
    w = np.asarray(work, np.int16)
    diff = np.abs(o - w).max(2).astype(np.float32)
    diff = cv2.GaussianBlur(diff, (0, 0), 2)
    changed = ndi.binary_opening(diff > thr, iterations=2)
    bm = box_mask_np(diff.shape, wbox)
    area = max(bm.sum(), 1)
    overflow = (changed & ~bm).sum() / area
    inside_change = (changed & bm).sum() / area

    # leftover outline: marker color still present on the outline band, beyond what the scene already had
    band = outline_band(diff.shape, wbox, width)
    col = np.array(MARKS[mark][0])
    near_out = (np.abs(o - col).sum(2) < 120) & band
    near_orig = (np.abs(w - col).sum(2) < 120) & band
    leftover = max(near_out.sum() - near_orig.sum(), 0) / max(band.sum(), 1)

    score = overflow + 3.0 * leftover + (1.0 if inside_change < 0.05 else 0.0)
    return dict(score=float(score), overflow=float(overflow), inside=float(inside_change),
                leftover=float(leftover), changed=changed, box=bm)


def build_region(changed, bm, grow=6, max_growth=3.0):
    """Blend region = box + changed pixels connected to it (keeps overflow), then dilated."""
    region = changed | bm
    lab, _ = ndi.label(region)
    keep = np.unique(lab[bm]); keep = keep[keep > 0]
    region = np.isin(lab, keep)
    if region.sum() > max_growth * bm.sum():
        region = bm.copy()
    return ndi.binary_dilation(region, iterations=grow)


# ---------- blending / matching ----------
def poisson_blend(src, dst, region):
    pad = 32
    s = cv2.copyMakeBorder(src, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    d = cv2.copyMakeBorder(dst, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    m = cv2.copyMakeBorder(region.astype(np.uint8) * 255, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0)
    ys, xs = np.where(m > 0)
    center = (int((xs.min() + xs.max()) // 2), int((ys.min() + ys.max()) // 2))
    res = cv2.seamlessClone(s, d, m, center, cv2.NORMAL_CLONE)
    return res[pad:-pad, pad:-pad]


def _lap_var(gray, mask):
    return float(cv2.Laplacian(gray, cv2.CV_32F)[mask].var())


def match_sharpness(gen, ref, region, max_blur=2.0):
    ring = ndi.binary_dilation(region, iterations=40) & ~ndi.binary_dilation(region, iterations=4)
    if ring.sum() < 100:
        return gen, 0.0
    g_ref = cv2.cvtColor(ref, cv2.COLOR_RGB2GRAY)
    g_gen = cv2.cvtColor(gen, cv2.COLOR_RGB2GRAY)
    target = np.log(_lap_var(g_ref, ring) + 1e-6)
    sigmas = [s for s in (0, 0.5, 0.8, 1.2, 1.6, 2.0, 2.5, 3.0) if s <= max_blur]
    def err(s):
        g = cv2.GaussianBlur(g_gen, (0, 0), s) if s > 0 else g_gen
        return abs(np.log(_lap_var(g, region) + 1e-6) - target)
    best = min(sigmas, key=err)
    return (cv2.GaussianBlur(gen, (0, 0), best) if best > 0 else gen), best


def jpeg_roundtrip(arr, q):
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, "JPEG", quality=q)
    return np.asarray(Image.open(buf).convert("RGB"))


# ---------- main ----------
@torch.no_grad()
def insert_object_drawn_box(
    pipe, image, bbox, obj,
    mark="red", prompt=None, line_width=None,
    n_candidates=6, seed=0,
    context_scale=2.5, target=None,
    steps=4, guidance=1.0,
    thr=28, grow=None, feather=8, max_growth=3.0,
    blend="poisson",                        # "poisson" or "alpha"
    match_quality=True, max_blur=2.0, jpeg_quality=None,
):
    """
    image : PIL image;  bbox : (x0, y0, x1, y1) pixels;  obj : text, e.g. "a white fish swimming left"
    mark  : outline style, one of MARKS ("red", "green", "magenta")
    Returns (final_image, debug_dict)
    """
    image = image.convert("RGB")
    x0, y0, x1, y1 = bbox
    assert 0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height, "bbox outside image"

    # 1) crop with context near native resolution
    crop_box = get_crop_box(bbox, image.size, context_scale)
    crop = image.crop(crop_box)
    cw, ch = crop.size
    ww, wh = work_size(cw, ch, target or auto_target(cw, ch))
    sx, sy = ww / cw, wh / ch
    wbox = ((x0 - crop_box[0]) * sx, (y0 - crop_box[1]) * sy,
            (x1 - crop_box[0]) * sx, (y1 - crop_box[1]) * sy)
    work = crop.resize((ww, wh), Image.LANCZOS)

    # 2) draw the box on the (otherwise untouched) scene
    width = line_width or max(3, round(0.008 * max(ww, wh)))
    marked = draw_marker(work, wbox, mark, width)
    prompt = prompt or default_prompt(obj, x1 - x0, y1 - y0, mark)
    grow = grow if grow is not None else width + 4      # make sure the region covers the outline

    # 3) best-of-N
    cands = []
    gdev = pipe.device if hasattr(pipe, "device") else "cpu"
    for i in range(n_candidates):
        gen = torch.Generator(device=gdev).manual_seed(seed + i)
        out = pipe(image=marked, prompt=prompt, height=wh, width=ww,
                   num_inference_steps=steps, guidance_scale=guidance, generator=gen).images[0].convert("RGB")
        if out.size != (ww, wh):
            out = out.resize((ww, wh), Image.LANCZOS)
        a = analyze_candidate(out, work, wbox, mark, width, thr)
        a.update(seed=seed + i, out=out)
        cands.append(a)
    best = min(cands, key=lambda c: c["score"])
    out, scrubbed = scrub_marker(best["out"], work, wbox, mark, width)   # remove any leftover outline pixels

    # 4) change-aware region at full crop resolution
    region_w = build_region(best["changed"], best["box"], grow, max_growth)
    region = cv2.resize(region_w.astype(np.uint8), (cw, ch), interpolation=cv2.INTER_NEAREST).astype(bool)
    gen_c = np.asarray(out.resize((cw, ch), Image.LANCZOS))
    crop_np = np.asarray(crop)

    blur_used = 0.0
    if match_quality:
        gen_c, blur_used = match_sharpness(gen_c, crop_np, region, max_blur)
    if jpeg_quality:
        gen_c = jpeg_roundtrip(gen_c, jpeg_quality)

    # 5) blend
    if blend == "poisson":
        try:
            new_crop = poisson_blend(gen_c, crop_np, region)
        except Exception as e:
            print("poisson failed, falling back to alpha:", e)
            blend = "alpha"
    if blend == "alpha":
        alpha = cv2.GaussianBlur(region.astype(np.float32), (0, 0), max(feather / 2, 0.1))[..., None]
        new_crop = (gen_c * alpha + crop_np * (1 - alpha)).round().astype(np.uint8)

    new_crop = np.where(ndi.binary_dilation(region, iterations=feather + 2)[..., None], new_crop, crop_np)
    final = image.copy()
    final.paste(Image.fromarray(new_crop.astype(np.uint8)), crop_box[:2])

    dbg = dict(prompt=prompt, mark=mark, scrubbed=scrubbed, crop_box=crop_box, marked_input=marked, raw_output=out, region=region,
               candidates=[{k: c[k] for k in ("seed", "score", "overflow", "inside", "leftover", "out")} for c in cands],
               chosen_seed=best["seed"], blur_used=blur_used)
    return final, dbg


# ---------- diagnostics: which box style does klein follow best? ----------
def compare_marks(pipe, image, bbox, obj, marks=None, seed=0, **kw):
    """One candidate per box style, raw outputs + finals side by side."""
    marks = marks or list(MARKS)
    results = {}
    for m in marks:
        results[m] = insert_object_drawn_box(pipe, image, bbox, obj, mark=m, n_candidates=1, seed=seed, **kw)
    fig, axs = plt.subplots(3, len(marks), figsize=(4 * len(marks), 10), squeeze=False)
    for j, m in enumerate(marks):
        final, d = results[m]
        axs[0, j].imshow(d["marked_input"]); axs[0, j].set_title(f"{m}: input", fontsize=10)
        axs[1, j].imshow(d["raw_output"]);   axs[1, j].set_title(f"{m}: raw output", fontsize=10)
        axs[2, j].imshow(final);             axs[2, j].set_title(f"{m}: final", fontsize=10)
        for i in range(3): axs[i, j].axis("off")
    plt.tight_layout(); plt.show()
    return results


# ---------- display ----------
def show(images, titles=None, figsize=(16, 6)):
    fig, axs = plt.subplots(1, len(images), figsize=figsize)
    axs = [axs] if len(images) == 1 else axs
    for i, (ax, im) in enumerate(zip(axs, images)):
        ax.imshow(im); ax.axis("off")
        if titles: ax.set_title(titles[i])
    plt.tight_layout(); plt.show()


def show_candidates(dbg, cols=6):
    cs = dbg["candidates"]
    cols = min(cols, len(cs)); rows = (len(cs) + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(3.2 * cols, 2.6 * rows), squeeze=False)
    for ax in axs.ravel(): ax.axis("off")
    for ax, c in zip(axs.ravel(), cs):
        ax.imshow(c["out"])
        chosen = c["seed"] == dbg["chosen_seed"]
        ax.set_title(f"seed {c['seed']}  score {c['score']:.2f}" + ("  ✓" if chosen else ""),
                     fontsize=9, color="green" if chosen else "black")
    plt.tight_layout(); plt.show()


def preview_bbox(image, bbox, color="red", width=4):
    im = image.convert("RGB").copy()
    ImageDraw.Draw(im).rectangle(bbox, outline=color, width=width)
    return im


def norm_to_px(bbox_norm, size):
    W, H = size
    return (bbox_norm[0] * W, bbox_norm[1] * H, bbox_norm[2] * W, bbox_norm[3] * H)

## Load your image and set the bounding box

`bbox` is `(x0, y0, x1, y1)` in **pixels** (use `norm_to_px` for normalized boxes). Pick a mark color that does **not** already appear in your scene.

In [ ]:
image = Image.open("scene.jpg").convert("RGB")     # <-- your image
bbox = (400, 300, 700, 600)                        # <-- your box
# bbox = norm_to_px((0.4, 0.5, 0.6, 0.8), image.size)

show([preview_bbox(image, bbox)], ["Input + bbox"], figsize=(8, 8))

## Step 1 (optional) — which box style does klein follow best?

In [ ]:
results = compare_marks(pipe, image, bbox, obj="a white fish swimming left", seed=0)

## Step 2 — full run

In [ ]:
MARK = "red"        # <-- the style that worked best above

result, dbg = insert_object_drawn_box(
    pipe, image, bbox,
    obj="a white fish swimming left",   # describe the object (and pose/direction if you care)
    mark=MARK,
    n_candidates=6,                     # best-of-N; each run is only 4 steps
    seed=0,
)

print("Prompt:", dbg["prompt"])
print(f"chosen seed {dbg['chosen_seed']}, sharpness blur sigma {dbg['blur_used']}")
show([image, dbg["marked_input"], dbg["raw_output"], result],
     ["Original", "Model input (crop, box drawn)", "Chosen raw output (crop)", "Final"])
show_candidates(dbg)
result.save("result_drawn_box.png")

## Tuning guide

| Symptom | Try |
|---|---|
| Box outline still visible in the output | another `mark` color; raise `n_candidates` (leftover outline is scored, lowest wins). As a last resort, marker-colored pixels left on the outline band are replaced by the original pixels (`dbg['scrubbed']` = fraction replaced), which only fixes the thin edge line, not marker color elsewhere |
| Object ignores the box / appears elsewhere | tighten the box, describe position in `obj`, raise `n_candidates`; if it keeps failing, approach 1 (filled box) is the stronger constraint |
| Object bigger than the box | overflow is scored and kept (not clipped); tighten the box or raise `n_candidates` |
| Model puts the box color on the object | pick a mark color absent from the scene (and from the object) |
| Nothing inserted | more candidates; rephrase `obj`; check `compare_marks` |
| Looks too crisp vs low-quality source | `max_blur=3.0`, `jpeg_quality=60`, or lower `target` (e.g. 512) |
| Seam / tint | `blend="alpha"` vs `"poisson"`; raise `grow` |
| `guidance_scale` rejected | remove it |